Lab 5 Shrikant_Gade 25215012111

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
pip uninstall bitsandbytes -y

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Note: you may need to restart the kernel to use updated packages.


In [4]:

pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

In [8]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [3]:
# def generate_response(prompt):
#     formatted_prompt = f"<s>[INST] {prompt} [/INST]"
    
#     inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
#     output = model.generate(
#         **inputs,
#         max_new_tokens=150
#     )
    
#     return tokenizer.decode(output[0], skip_special_tokens=True)

def generate_response(prompt):
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [4]:
mode = input("Enter mode (zero/few): ").strip().lower()

if mode == "zero":
    prompt = "Explain Graph Neural Networks in one paragraph."
    output = generate_response(prompt)
    print("\nZero-Shot Output:\n")
    print(output)

elif mode == "few":
    few_shot_prompt = """
Translate English to French:

English: Hello, how are you?
French: Bonjour, comment ça va?

English: I love machine learning.
French: J'aime l'apprentissage automatique.

English: This is a beautiful day.
French:
"""
    output = generate_response(few_shot_prompt)
    print("\nFew-Shot Output:\n")
    print(output)

else:
    print("Invalid option")

Enter mode (zero/few):  zero


NameError: name 'tokenizer' is not defined

In [24]:
zero_shot_prompt = "Explain Graph Neural Networks in one paragraph."
print(generate_response(zero_shot_prompt))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] Explain Graph Neural Networks in one paragraph. [/INST] Graph Neural Networks (GNNs) are a type of neural network designed to learn from graph data, where nodes represent entities and edges represent relationships between them. GNNs extend traditional neural networks by incorporating graph structure into the model, enabling them to learn and reason about complex relationships between data points. In contrast to other machine learning models that require data to be preprocessed into a fixed, flat structure, such as vectors or matrices, GNNs can directly process graph data, making them particularly useful for applications where the data has a natural graph structure, such as social networks, chemical compounds, or recommendation systems. GNNs operate by propagating information between nodes through messages passed along edges, allowing them to capture both local


In [22]:
few_shot_prompt = """
Translate English to French:

English: Hello, how are you?
French: Bonjour, comment ça va?

English: I love machine learning.
French: J'aime l'apprentissage automatique.

English: This is a beautiful day.
French:
"""

In [23]:
print(generate_response(few_shot_prompt))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] 
Translate English to French:

English: Hello, how are you?
French: Bonjour, comment ça va?

English: I love machine learning.
French: J'aime l'apprentissage automatique.

English: This is a beautiful day.
French:
 [/INST] C'est un beau jour. (This is a beautiful day.)

English: What is your name?
French: Comment vous appele-t-on?

English: How old are you?
French: Quel âge avez-vous?

English: Where are you from?
French: D'où êtes-vous?

English: Can you help me, please?
French: Pouvez-vous m'aider, s'il vous plaît?

English: Thank you very much.
French: Merci beaucoup.

English: You're welcome.
French:


In [2]:
import tkinter as tk
from tkinter import ttk, scrolledtext
def run_model():
    output_box.delete("1.0", tk.END)

    mode = mode_var.get()

    if mode == "Zero-Shot":
        user_prompt = input_box.get("1.0", tk.END).strip()

        if not user_prompt:
            output_box.insert(tk.END, "Please enter a prompt.")
            return

        response = generate_response(user_prompt)
        output_box.insert(tk.END, response)

    elif mode == "Few-Shot":

        few_shot_prompt = """
Translate English to French:

English: Hello, how are you?
French: Bonjour, comment ça va?

English: I love machine learning.
French: J'aime l'apprentissage automatique.

English: This is a beautiful day.
French:
"""
        response = generate_response(few_shot_prompt)
        output_box.insert(tk.END, response)


# -------------------- TKINTER WINDOW --------------------

root = tk.Tk()
root.title("Mistral LLM GUI - Zero Shot / Few Shot")
root.geometry("800x600")

# Mode selection
mode_var = tk.StringVar(value="Zero-Shot")

mode_label = ttk.Label(root, text="Select Mode:")
mode_label.pack(pady=5)

mode_dropdown = ttk.Combobox(root, textvariable=mode_var, values=["Zero-Shot", "Few-Shot"])
mode_dropdown.pack(pady=5)

# Input box (for Zero-Shot)
input_label = ttk.Label(root, text="Enter Prompt (Only for Zero-Shot):")
input_label.pack(pady=5)

input_box = scrolledtext.ScrolledText(root, height=6)
input_box.pack(padx=10, pady=5, fill="x")

# Run button
run_button = ttk.Button(root, text="Generate", command=run_model)
run_button.pack(pady=10)

# Output box
output_label = ttk.Label(root, text="Model Output:")
output_label.pack(pady=5)

output_box = scrolledtext.ScrolledText(root, height=15)
output_box.pack(padx=10, pady=5, fill="both", expand=True)

root.mainloop()

TclError: no display name and no $DISPLAY environment variable